In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
"""
=============================================================
 STEP 3 — ML Training & Prediction
 Microsoft Fabric Notebook — reads/writes Lakehouse Tables
=============================================================
Reads  : ml_features  (Delta table from Notebook 02)
Writes : stockout_predictions  (Delta table → Power BI)
Tracks : MLflow experiments + Fabric Model Registry
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer, StandardScaler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient

spark = SparkSession.builder.appName("StockoutML").getOrCreate()

# ── Table names ────────────────────────────────────────────
TABLE_FEATURES    = "ml_features"          # input  — from Notebook 02
TABLE_PREDICTIONS = "stockout_predictions" # output — for Power BI

TARGET     = "days_until_stockout"
EXPERIMENT = "stockout_prediction_fmcg"

NUM_FEATURES = [
    "stock_on_hand","safety_stock","reorder_point",
    "below_safety_stock_flag","days_of_supply",
    "stock_to_safety_ratio","stock_vs_reorder","projected_cover_days",
    "replenishment_qty","lead_time_days",
    "avg_sales_7d","avg_sales_14d","avg_sales_30d",
    "sum_sales_7d","max_sales_7d","std_sales_7d",
    "promo_count_7d","festival_count_7d",
    "units_sold_lag1","units_sold_lag3","units_sold_lag7",
    "stock_lag1","stock_lag3","stock_lag7",
    "soh_delta_1d","soh_delta_3d","soh_delta_7d",
    "day_of_week","week_of_year","month","quarter",
    "is_weekend","is_festival","is_month_end",
    "category_code","is_perishable","shelf_life_days",
]

# ══════════════════════════════════════════════════════════
# LOAD FEATURE TABLE FROM LAKEHOUSE
# ══════════════════════════════════════════════════════════
df = spark.read.table(TABLE_FEATURES).dropna(subset=[TARGET])

print(f"Loaded table '{TABLE_FEATURES}': {df.count():,} rows | {len(NUM_FEATURES)} features")

print("\nRows per category:")
df.groupBy("category").agg(
    F.count("*").alias("rows"),
    F.round(F.avg(TARGET), 1).alias("avg_days_until_stockout")
).orderBy("category").show()

print("Risk band distribution:")
df.select(
    F.when(F.col(TARGET) <= 7,  "CRITICAL (<=7d)")
     .when(F.col(TARGET) <= 14, "WARNING (8-14d)")
     .otherwise("OK (>14d)").alias("risk_band")
).groupBy("risk_band").count().orderBy("risk_band").show()

# ══════════════════════════════════════════════════════════
# TIME-BASED TRAIN / TEST SPLIT
# Train: Jan–Oct 2024  |  Test: Nov–Dec 2024
# ══════════════════════════════════════════════════════════
train = df.filter(F.col("date") < "2024-11-01")
test  = df.filter(F.col("date") >= "2024-11-01")
print(f"Train: {train.count():,} rows  |  Test: {test.count():,} rows")

# ══════════════════════════════════════════════════════════
# SHARED PIPELINE STAGES
# ══════════════════════════════════════════════════════════
imp_cols  = [f"{c}_imp" for c in NUM_FEATURES]
imputer   = Imputer(inputCols=NUM_FEATURES, outputCols=imp_cols, strategy="median")
assembler = VectorAssembler(inputCols=imp_cols, outputCol="raw_features", handleInvalid="skip")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
evaluator = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction")

mlflow.set_experiment(EXPERIMENT)

def run_experiment(model_obj, params, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({**params, "train_cutoff": "2024-11-01", "n_features": len(NUM_FEATURES)})
        pipe   = Pipeline(stages=[imputer, assembler, scaler, model_obj])
        fitted = pipe.fit(train)
        preds  = fitted.transform(test)
        rmse = evaluator.setMetricName("rmse").evaluate(preds)
        mae  = evaluator.setMetricName("mae").evaluate(preds)
        r2   = evaluator.setMetricName("r2").evaluate(preds)
        mlflow.log_metrics({"RMSE": rmse, "MAE": mae, "R2": r2})
        mlflow.spark.log_model(fitted, run_name)
        print(f"\n{run_name}  →  RMSE: {rmse:.2f}d | MAE: {mae:.2f}d | R²: {r2:.4f}")
        return run.info.run_id, rmse, fitted, preds

# ══════════════════════════════════════════════════════════
# MODEL 1 — Gradient Boosted Trees
# ══════════════════════════════════════════════════════════
gbt = GBTRegressor(featuresCol="features", labelCol=TARGET,
                   maxIter=100, maxDepth=5, stepSize=0.1, seed=42)
gbt_run_id, gbt_rmse, gbt_model, gbt_preds = run_experiment(
    gbt, {"model":"GBTRegressor","maxIter":100,"maxDepth":5,"stepSize":0.1}, "GBT_v1"
)

# ══════════════════════════════════════════════════════════
# MODEL 2 — Random Forest
# ══════════════════════════════════════════════════════════
rf = RandomForestRegressor(featuresCol="features", labelCol=TARGET,
                           numTrees=100, maxDepth=6, seed=42)
rf_run_id, rf_rmse, rf_model, rf_preds = run_experiment(
    rf, {"model":"RandomForestRegressor","numTrees":100,"maxDepth":6}, "RF_v1"
)

# ══════════════════════════════════════════════════════════
# SELECT BEST MODEL
# ══════════════════════════════════════════════════════════
if gbt_rmse <= rf_rmse:
    best_model, best_preds, best_run_id, best_name = gbt_model, gbt_preds, gbt_run_id, "GBT_v1"
    importances = best_model.stages[-1].featureImportances.toArray()
else:
    best_model, best_preds, best_run_id, best_name = rf_model,  rf_preds,  rf_run_id,  "RF_v1"
    importances = best_model.stages[-1].featureImportances.toArray()

print(f"\nBest model: {best_name}  (RMSE = {min(gbt_rmse, rf_rmse):.2f} days)")

print("\nTop 12 Feature Importances:")
feat_imp = sorted(
    zip([c.replace("_imp","") for c in imp_cols], importances),
    key=lambda x: x[1], reverse=True
)
for feat, imp in feat_imp[:12]:
    print(f"  {feat:<35} {imp:.4f}  {'█' * int(imp * 80)}")

# ══════════════════════════════════════════════════════════
# REGISTER BEST MODEL IN FABRIC MODEL REGISTRY
# ══════════════════════════════════════════════════════════
client     = MlflowClient()
model_name = "StockoutPredictor_FMCG"

try:
    client.create_registered_model(model_name)
except Exception:
    pass  # already exists

mv = client.create_model_version(
    name=model_name,
    source=f"runs:/{best_run_id}/{best_name}",
    run_id=best_run_id
)
client.transition_model_version_stage(name=model_name, version=mv.version, stage="Production")
print(f"\nRegistered: '{model_name}' v{mv.version} → Production stage")

# ══════════════════════════════════════════════════════════
# SCORE FULL DATASET + SAVE TO LAKEHOUSE TABLE
# ══════════════════════════════════════════════════════════
all_preds = best_model.transform(df)

output = all_preds.select(
    "date","sku_id","sku_name","category","store_id",
    "stock_on_hand","safety_stock","reorder_point","lead_time_days",
    F.col(TARGET).alias("actual_days_until_stockout"),
    F.greatest(F.round("prediction", 1), F.lit(0.0)).alias("predicted_days_until_stockout"),
    F.when(F.col("prediction") <= 7,  "CRITICAL")
     .when(F.col("prediction") <= 14, "WARNING")
     .otherwise("OK")
     .alias("risk_band"),
    F.when(F.col("prediction") <= F.col("lead_time_days"), "REORDER_NOW")
     .otherwise("MONITOR")
     .alias("procurement_action"),
)

# Write to Lakehouse Delta table (Power BI DirectLake ready)
output.write.mode("overwrite").format("delta").saveAsTable(TABLE_PREDICTIONS)

print(f"\nPredictions written to table: '{TABLE_PREDICTIONS}'")
print("This table is now available in Power BI via DirectLake mode.\n")

output.groupBy("risk_band").count().orderBy("risk_band").show()

# ── Latest-day snapshot for ops team ──────────────────────
latest   = output.agg(F.max("date")).collect()[0][0]
snapshot = output.filter(F.col("date") == latest)

print(f"Risk snapshot for {latest}:")
snapshot.groupBy("category","risk_band").count().orderBy("category","risk_band").show(30)

print("CRITICAL SKUs — immediate action required:")
snapshot.filter(F.col("risk_band") == "CRITICAL") \
    .select("sku_id","sku_name","category","store_id",
            "predicted_days_until_stockout","procurement_action") \
    .orderBy("predicted_days_until_stockout") \
    .show(20, truncate=False)


StatementMeta(, 15d5bd96-51e5-4391-a0bc-8909625d144f, 3, Finished, Available, Finished, False)

Loaded table 'ml_features': 27,450 rows | 37 features

Rows per category:
+-------------+----+-----------------------+
|     category|rows|avg_days_until_stockout|
+-------------+----+-----------------------+
|    Beverages|4392|                   18.0|
|        Dairy|3660|                   17.2|
| Frozen Foods|1830|                   20.7|
|    Household|3660|                   22.0|
|Personal Care|4392|                   18.5|
|       Snacks|4026|                   16.7|
|      Staples|5490|                   17.3|
+-------------+----+-----------------------+

Risk band distribution:
+---------------+-----+
|      risk_band|count|
+---------------+-----+
|CRITICAL (<=7d)| 6239|
|      OK (>14d)|15531|
|WARNING (8-14d)| 5680|
+---------------+-----+

Train: 22,875 rows  |  Test: 4,575 rows

GBT_v1  →  RMSE: 1.00d | MAE: 0.56d | R²: 0.9943


2026/03/28 17:09:01 INFO mlflow.tracking.fluent: Experiment with name 'stockout_prediction_fmcg' does not exist. Creating a new experiment.
2026/03/28 17:10:46 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpkbbzjofm/model, flavor: spark). Fall back to return ['pyspark==3.5.1.5.4.20240407']. Set logging level to DEBUG to see the full traceback. 
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


2026/03/28 17:11:49 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpu0_jej5r/model, flavor: spark). Fall back to return ['pyspark==3.5.1.5.4.20240407']. Set logging level to DEBUG to see the full traceback. 



RF_v1  →  RMSE: 1.75d | MAE: 1.00d | R²: 0.9827



Best model: GBT_v1  (RMSE = 1.00 days)

Top 12 Feature Importances:
  projected_cover_days                0.8815  ██████████████████████████████████████████████████████████████████████
  stock_on_hand                       0.0205  █
  avg_sales_7d                        0.0166  █
  days_of_supply                      0.0110  
  safety_stock                        0.0105  
  units_sold_lag3                     0.0092  
  sum_sales_7d                        0.0085  
  stock_lag1                          0.0073  
  week_of_year                        0.0045  
  stock_lag3                          0.0039  
  units_sold_lag7                     0.0036  
  max_sales_7d                        0.0032  

Registered: 'StockoutPredictor_FMCG' v1 → Production stage

Predictions written to table: 'stockout_predictions'
This table is now available in Power BI via DirectLake mode.

+---------+-----+
|risk_band|count|
+---------+-----+
| CRITICAL| 5948|
|       OK|15677|
|  WARNING| 5825|
+---------+